<a href="https://colab.research.google.com/github/dakshigoel22/BIG_DATA_606/blob/main/605_hw3_dakshigoel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Vector databases and Chroma

○ Explain what kind of datastore vector databases are.

A vector database is a specialized datastore designed to store and query high-dimensional vector embeddings.

Embeddings: When you give an AI model text, an image, or audio, it converts that data into a long list of numbers (a vector). This vector represents the "semantic meaning" of the data.

Vector Space: The database places these vectors in a multi-dimensional space. Items that are conceptually similar (e.g., "dog" and "puppy") are mathematically placed close to each other.

Similarity Search: Instead of using SQL WHERE clauses, these databases use algorithms like HNSW (Hierarchical Navigable Small World) to find the "nearest neighbors" to a query.

○ Why/when would you use them?

We use a vector database when your data is unstructured (text, images, video) and you need to find things based on intent rather than keywords.

RAG (Retrieval-Augmented Generation): To give an LLM access to your private documents (PDFs, emails) so it can answer questions accurately without hallucinating.

Recommendation Systems: Finding products similar to what a user just bought based on visual or stylistic features.

Semantic Search: Building a search bar that understands that "How do I fix my car?" is related to "Automotive repair guides."

Anomaly Detection: Finding data points that are "mathematically far" from everything else in the system.

○ What is Chroma?

Chroma (often called ChromaDB) is an open-source vector database specifically designed to be the simplest way to build AI applications. It brands itself as the "AI-native" database because it focuses entirely on the developer experience of connecting data to LLMs.

○ How does Chroma compare to other vector databases like Milvus, Weaviate, and
Pinecone?

■ What are the strengths and weaknesses of each?
| Database     | Type                    | Strenght        |  Weaknesses            |
| ------------ | ----------------------- | ----------------------- | ------------------------ |
| **Chroma**   | Open-source, embedded   | Prototyping, small apps | Not great at large scale |
| **Pinecone** | Fully managed SaaS      | Production apps         | Expensive, less control  |
| **Milvus**   | Distributed open-source | Billion-scale systems   | Complex setup            |
| **Weaviate** | Open-source + cloud     | Hybrid search apps      | Learning curve           |



○ What are the different modes you can run Chroma in?

I can run Chroma in three main ways depending on your project stage:

1. In-Memory (Ephemeral): Everything is stored in RAM. Data is lost when the script stops. Perfect for unit tests or quick scripts.

2. Persistent (Local): Data is saved to a local folder on your hard drive (using SQLite). This is the standard for most hobbyist/small projects.

3. Client/Server (Distributed): You run Chroma as a separate service (often in a Docker container). Your application connects to it via an API. This mode allows multiple applications to share the same database and is the path toward production.

#3. Setup

In [ ]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
#creating a chroma client
import chromadb
chroma_client = chromadb.Client()

#4. Data Creation

In [ ]:
# Documents: 8 specific facts about Skillify
doc1 = "Skillify is a dedicated platform to discover, book, and build profiles for creative skills."
doc2 = "Users can search for dance workshops by dance style, skill level, location, and date."
doc3 = "The platform allows dancers to follow their favorite choreographers and stay updated on events."
doc4 = "For choreographers and studios, Skillify provides visibility and structured event listings."
doc5 = "Skillify offers seamless booking management for organizers and instant booking for users."
doc6 = "The mission of Skillify is to help people grow, elevate their skills, and stay organized."
doc7 = "Dancers can build their own professional dance profiles directly on the Skillify app."
doc8 = "Skillify currently focuses on discovering dance workshops and connecting with choreographers."

# Queries: 2 questions a user might actually ask
query = "What is Skillify?"
query1 = "How can I find a hip-hop class near me this weekend?"
query2 = "What tools does Skillify offer for dance instructors and studio owners?"

all_docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8]

In [ ]:
#create a collection using Create (or get) the collection called "answers"
collection = chroma_client.get_or_create_collection(name="answers")

#adding the documents to the collection

collection.add(
    documents=all_docs,
    ids=["id1", "id2", "id3", "id4", "id5", "id6", "id7", "id8"]
)

print(f"Succusfully added {collection.count()} documents to the 'answers' collection")

Succusfully added 8 documents to the 'answers' collection


#5. Embedding Creation

**○ Explain what it means to create vector embeddings, and why we do it.**

Creating vectors means transforming unstructured data - text in our case into numerical vectors.
By converting them, it helps to caputre semantic meaning, allowing AI models to measure similarity and understand relationship between the data.

**○ What is an embedding space?**

An embedding space is a multi-dimensional "map" where every piece of data is assigned a specific coordinate.Unlike a 2D map ($x, y$) or a 3D space ($x, y, z$), an embedding space often has hundreds or thousands of dimensions. In this space, the distance between two points (calculated using formulas like Cosine Similarity) represents how related those two concepts are.


○ When you used ` .add ` in the previous step, Chroma automatically created embeddings for your strings.

■ What embedding algorithm did Chroma use for this by default ?

Chroma uses the all-MiniLM-L6-v2 model from the sentence-transformers library to create embeddings.

■ Briefly explain what this algorithm is and what it is doing in 1-3 sentences.

- all-MiniLM-L6-v2 is a pre-trained, high-performance Transformer-based model optimized to convert sentences or short documents into dense 384-dimensional vectors.
- What it does: It maps text into a semantic vector space where similar texts are positioned close together, enabling Chroma to perform efficient semantic searches based on meaning rather than exact keyword matches.
- How it runs: The algorithm runs locally on the machine—often via ONNX Runtime—meaning it does not require API keys or external calls to cloud providers.

In [ ]:
print("Number of documents :", collection.count())

Number of documents : 8


In [ ]:
test_vector = collection._embedding_function(["test"])
dimensionality = len(test_vector[0])
print(dimensionality)

384


**● What does this number represent?**

This number representsthe dimensions of the embedding vectors.

**● Does it match the number you expect based on the embedding algorithm you identified earlier? Why or why not**

Yes, it matches the expected number as the embedding algorythm used by Chroma 'all-MiniLM-L6-v2' converts the short documents into dense 384-dimensional vectors.

#6. Index Creation

○ **Explain what indexing is, in the context of vector databases.**

In a vector database, indexing is the process of organizing high-dimensional vectors into a searchable structure. Without an index, the database would have to perform a "brute-force" search, comparing the query to every single item in the database one by one, which is incredibly slow for large datasets.


■ **What does the database do when it creates an index?**

When creating an index, the database maps the vectors into a mathematical structure that groups "similar" items together before you ever run a search.

- Clustering/Partitioning: It divides the vector space into regions or clusters.

- Pathfinding: It creates a "navigation mesh" (like in the HNSW algorithm), where data points act as nodes in a graph.

- Dimensionality Reduction: It may compress the vectors (Product Quantization) to make them take up less memory while preserving their relative distance.


○ **What are the main tradeoffs for precomputing vector indexes?**

While indexing makes searching fast, it comes with three primary costs:

- **Accuracy vs. Speed (Recall):** Most vector indexes are based on ANN. To get massive speed, the database might occasionally miss the absolute closest neighbor in favor of one that is "close enough" and found much faster.

- **Memory Overhead:** Storing the index requires significant RAM or disk space on top of the raw data.

- **Inversion/Update Cost:** Every time we add or delete a document (like adding a new workshop to Skillify), the database has to re-calculate parts of the index. This makes "writes" slower than a simple storage system.

○  When you used .add in the step 4, Chroma first wrote to a Write-Ahead-Log.We covered the concept of a WAL when we talked about Wide-Column
Databases. There is also a useful Chroma discussion here (based on an
older version, but it works mostly the same today and this is better than
their architecture documentation, in my opinion).

■ Explain: What is a WAL and why is it useful? What is Chroma doing when it writes to the WAL?


A Write-Ahead-Log (WAL) is a file-based log that records all data changes before they are applied to the main database, ensuring durability and data integrity in case of crashes. It is useful for recovering data by replaying logged operations after a failure. Chroma uses it for persistence, writing operations (like embeddings and metadata) to disk before updating memory, ensuring data is not lost.

Chroma WAL Action: When calling .add, Chroma first writes the new data (embeddings and metadata) sequentially to the WAL. This fast, sequential write guarantees that even if the process crashes, the data can be recovered by replaying the log upon restart.

○ Chroma also automatically created an index for you.

■ What algorithm did it use to build the index ?
Briefly explain in your own words what this algorithm is and what it is doing in 1-3 sentences.


Chroma uses HNSW (Hierarchical Navigable Small World).
HNSW builds a multi-layered graph index for fast, ANN searching. HNSW builds a graph of nodes, where each node is a vector. Nearby nodes are connected to form a searchable network.

HNSW converts linear, expensive searches (low scalability) into logarithmic time, enabling quick retrieval from millions of vectors.

#7. Similarity Search


○ Explain what similarity search is in the context of vector databases.

In the context of vector databases, similarity search (or Nearest Neighbor search) is the process of finding the data points in a high-dimensional space that are mathematically "closest" to a target point.  Similarity search looks for conceptual proximity. It calculates the distance between vectors using metrics like Cosine Similarity or Euclidean Distance.

○ What does it mean to embed a query?

To embed a query means to take the user's raw input string and run it through the exact same machine learning model used to process the original documents (in Chroma’s case, all-MiniLM-L6-v2). This turns the queries into a vector of 384 numbers.



○ Perform a similarity search by running .query for each of your 2 query strings
from step 4 and retrieving the top 2 documents (more info here)

○ Print the documents that were returned.

○ What do these results represent? How would you use them to respond to a user
of your chatbot?


In [ ]:
# Similarity search by running .query for each of your 2 query strings

# Query 1: "How can I find a hip-hop class near me this weekend?"
results1 = collection.query(
    query_texts=[query1],
    n_results=2
)

# Query 2: "What tools does Skillify offer for dance instructors and studio owners?"
results2 = collection.query(
    query_texts=[query2],
    n_results=2
)

# Printing the retrieved documents
print(f"Results for Query 1 ('{query1}'):")
for i, doc in enumerate(results1['documents'][0]):
    print(f"  Result {i+1}: {doc}")

print(f"\nResults for Query 2 ('{query2}'):")
for i, doc in enumerate(results2['documents'][0]):
    print(f"  Result {i+1}: {doc}")

Results for Query 1 ('How can I find a hip-hop class near me this weekend?'):
  Result 1: Users can search for dance workshops by dance style, skill level, location, and date.
  Result 2: Skillify currently focuses on discovering dance workshops and connecting with choreographers.

Results for Query 2 ('What tools does Skillify offer for dance instructors and studio owners?'):
  Result 1: Skillify currently focuses on discovering dance workshops and connecting with choreographers.
  Result 2: For choreographers and studios, Skillify provides visibility and structured event listings.


## Interpretation of results

**What do these results represent? How would you use them to respond to a user of your chatbot?**

These results represent the top-k most semantically relevant context found in the database (out of 8 docs). For Query 1, the database likely returned the document about searching by "style, location, and date" and the document about "discovering dance workshops."

In a production chatbot, you wouldn't just dump these raw strings to the user. Instead, you would use them as Context for a Large Language Model (like Gemini or GPT-4). This is part of the RAG (Retrieval-Augmented Generation) workflow.

# Scale
○ Now imagine your company has grown significantly. You have acquired every competitor in the space and even unrelated companies, and now the products
and services you offer have greatly expanded. The kinds of data you need to represent has also expanded (images, videos, gifs, songs, etc).

**○ What are your options for scaling your vector database / chatbot?**


1. **Vertical Scaling:** With Chroma or an open-source instance,  move it to a high-memory server with many CPU cores and high-speed NVMe storage. This is the simplest to implement but has a hard ceiling once the index exceeds the RAM of a single machine.

2. **Horizontal Scaling (The "Cluster" approach):** We can migrate to a distributed database like Milvus or Weaviate. These systems split collection into shards (smaller pieces) across multiple servers. When a user searches for a song or a dance video, the query is sent to all servers simultaneously, and the results are aggregated.

3. **Managed Serverless Scaling :** I can move to a service like Pinecone or Google Cloud Vertex AI Search. These services automatically handle the "sharding" and "replication".

**○ What considerations/tradeoffs do you need to weigh?**

1. Latency vs. Accuracy : To keep the chatbot fast (low latency) with millions of videos, we will have to use Product Quantization (PQ), which compresses vectors. This makes search lightning-fast but might slightly decrease the ""relevance"" of the search results.

2. Cost vs. Complexity:  Managed services are expensive but allow your team to focus purely on the  product features.

3. Consistency vs. Availability: In a global system, if a choreographer in Tokyo uploads a video, how soon should a student in New York see it in the chatbot? Ensuring ""Immediate Consistency"" across the globe is technically difficult and can slow down the system.